In [1]:
import pandas as pd

station = pd.read_parquet("../../Data/sort_data/2024_data.parquet")

In [2]:
# 특정 정류소가 시작이거나 종료인 데이터만 반환
def filter_station_rows(df, keyword):
    mask = (
        df['시작_대여소_ID'].astype(str).str.contains(keyword, na=False) |
        df['종료_대여소_ID'].astype(str).str.contains(keyword, na=False)
    )
    
    filtered_df = df[mask].copy()
    
    return filtered_df

In [3]:
# 종료가 해당 정류소인 경우 도착시간으로 변경
import numpy as np

def update_end(df, station_id):
    mask = df["종료_대여소_ID"] == station_id

    # Timedelta -> 시간(실수)
    hours = df.loc[mask, "전체_이용_분"].dt.total_seconds() / 3600.0

    # 버림 처리
    raw = np.floor(df.loc[mask, "시간대"] - hours).astype(int)

    # 음수 시간 처리: 기준_날짜 하루(또는 여러 일) 줄이고 시간대 보정
    day_shift = np.floor_divide(raw, 24)  # 음수면 -1, -2 ...
    adj_hour = raw - day_shift * 24       # 0~23로 보정

    # 날짜 업데이트
    df.loc[mask, "기준_날짜"] = (
        pd.to_datetime(df.loc[mask, "기준_날짜"]) + pd.to_timedelta(day_shift, unit="D")
    )

    # 시간대 업데이트
    df.loc[mask, "시간대"] = adj_hour

    # 집계 기준 변경
    df.loc[mask, "집계_기준"] = "도착시간"

    # 기준_날짜 -> 시간대 정렬
    df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)
    return df


In [4]:
# 출퇴근시간, 요일, 주말 컬럼 추가
def add_time_flags(df, commute_hours=None):
    df = df.copy()

    # 출퇴근 시간대 기본값
    if commute_hours is None:
        commute_hours = [7, 8, 9, 17, 18, 19]

    # 날짜 파싱
    date = pd.to_datetime(df["기준_날짜"])

    # 요일 컬럼 (월~일)
    weekday_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
    df["요일"] = date.dt.weekday.map(weekday_map)

    # 출퇴근시간 컬럼
    df["출퇴근시간"] = df["시간대"].isin(commute_hours).astype(int)

    # 2024년 공휴일 (한국)
    holidays_2024 = pd.to_datetime([
        "2024-01-01",
        "2024-02-09", "2024-02-10", "2024-02-11", "2024-02-12",
        "2024-03-01",
        "2024-04-10",
        "2024-05-05", "2024-05-06",
        "2024-05-15",
        "2024-06-06",
        "2024-08-15",
        "2024-09-16", "2024-09-17", "2024-09-18",
        "2024-10-01",
        "2024-10-03",
        "2024-10-09",
        "2024-12-25",
    ])

    # 주말 컬럼 (토/일 또는 2024 공휴일)
    is_weekend = date.dt.weekday >= 5
    is_holiday_2024 = date.isin(holidays_2024)
    df["주말"] = (is_weekend | is_holiday_2024).astype(int)

    return df


In [5]:
# 모델용 피처 생성
def add_model_features(df):
    df = df.copy()

    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"], errors="coerce")
    df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)

    df["dow"] = df["기준_날짜"].dt.dayofweek
    df["day_type"] = (df["dow"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["시간대"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["시간대"] / 24.0)

    df["lag1"] = df["전체_건수"].shift(1)
    df["lag3"] = df["전체_건수"].shift(3)
    df["lag24"] = df["전체_건수"].shift(24)
    df["lag48"] = df["전체_건수"].shift(48)
    df["lag168"] = df["전체_건수"].shift(168)

    # 누수 방지: shift(1) 이후 rolling
    df["rolling3"] = df["전체_건수"].shift(1).rolling(3).mean()
    df["rolling24"] = df["전체_건수"].shift(1).rolling(24).mean()

    df["rush_hour"] = (((df["시간대"] >= 7) & (df["시간대"] <= 9)) | ((df["시간대"] >= 17) & (df["시간대"] <= 19))).astype(int)

    return df


In [6]:
st_2247 = filter_station_rows(station, "ST-2247")
st_2252 = filter_station_rows(station, "ST-2252")
st_2425 = filter_station_rows(station, "ST-2425")

In [7]:
st_2247 = update_end(st_2247, "ST-2247")
st_2252 = update_end(st_2252, "ST-2252")
st_2425 = update_end(st_2425, "ST-2425")

/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_1908/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[13 18 17 ... 12 13  6]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_1908/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[16 16 16 ... 14 20 13]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_1908/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[15 15 15 ...  0 18 10]' has dtype incompatible with int32, please explicitly

In [8]:
st_2247 = add_time_flags(st_2247)
st_2252 = add_time_flags(st_2252)
st_2425 = add_time_flags(st_2425)

In [9]:
# 피처 적용
st_2247 = add_model_features(st_2247)
st_2252 = add_model_features(st_2252)
st_2425 = add_model_features(st_2425)

features_hgb = [
    '시간대',
    'day_type',
    '온도',
    '습도',
    '불쾌지수',
    '강수량',
    '적설량',
    'lag1',
    'lag3',
    'lag24',
    'hour_sin',
    'hour_cos',
    'rolling3',
    'rolling24',
    'lag48',
    'lag168',
    'dow',
    'rush_hour'
]


In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 필요한 경우 피처 생성 함수 보강
if 'add_model_features' not in globals():
    def add_model_features(df):
        df = df.copy()
        df["기준_날짜"] = pd.to_datetime(df["기준_날짜"], errors="coerce")
        df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)
        df["dow"] = df["기준_날짜"].dt.dayofweek
        df["day_type"] = (df["dow"] >= 5).astype(int)
        df["hour_sin"] = np.sin(2 * np.pi * df["시간대"] / 24.0)
        df["hour_cos"] = np.cos(2 * np.pi * df["시간대"] / 24.0)
        df["lag1"] = df["전체_건수"].shift(1)
        df["lag3"] = df["전체_건수"].shift(3)
        df["lag24"] = df["전체_건수"].shift(24)
        df["lag48"] = df["전체_건수"].shift(48)
        df["lag168"] = df["전체_건수"].shift(168)
        df["rolling3"] = df["전체_건수"].shift(1).rolling(3).mean()
        df["rolling24"] = df["전체_건수"].shift(1).rolling(24).mean()
        df["rush_hour"] = (((df["시간대"] >= 7) & (df["시간대"] <= 9)) | ((df["시간대"] >= 17) & (df["시간대"] <= 19))).astype(int)
        return df

def train_model2(df, target_col="전체_건수", features=None, fallback_test_ratio=0.2):
    df = df.copy()
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"], errors="coerce")

    # 필요한 피처가 없으면 생성 함수 사용
    if "lag168" not in df.columns or "rush_hour" not in df.columns:
        df = add_model_features(df)

    if features is None:
        features = [
            '시간대',
            'day_type',
            '온도',
            '습도',
            '불쾌지수',
            '강수량',
            '적설량',
            'lag1',
            'lag3',
            'lag24',
            'hour_sin',
            'hour_cos',
            'rolling3',
            'rolling24',
            'lag48',
            'lag168',
            'dow',
            'rush_hour'
        ]

    # 결측 제거
    df = df.dropna(subset=features + [target_col])

    train = df[df['기준_날짜'].dt.year == 2024]
    test  = df[df['기준_날짜'].dt.year == 2025]

    # 2025 데이터가 없으면 최신 구간으로 테스트 분할
    if len(test) == 0:
        df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)
        n_test = int(len(df) * fallback_test_ratio)
        if n_test == 0:
            raise ValueError("테스트 샘플이 없습니다. 데이터가 너무 적습니다.")
        train = df.iloc[:-n_test]
        test = df.iloc[-n_test:]
        print(f"[warn] 2025 데이터가 없어 최근 {n_test}개를 테스트로 사용합니다.")

    X_train = train[features]
    y_train = train[target_col]

    X_test = test[features]
    y_test = test[target_col]

    hgb = HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=600,
        max_depth=10,
        min_samples_leaf=10,
        random_state=42
    )

    hgb.fit(X_train, y_train)
    pred_hgb = hgb.predict(X_test)

    print("HGB MAE:", mean_absolute_error(y_test, pred_hgb))
    print("HGB RMSE:", np.sqrt(mean_squared_error(y_test, pred_hgb)))
    print("HGB R2:", r2_score(y_test, pred_hgb))

    return hgb


--------
#### 기존 학습

In [11]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_model(
    df,
    time_col=("기준_날짜", "시간대"),
    feature_cols=("시간대","주말", "출퇴근시간", "온도", "습도", "강수량", "적설량"),
    target_col="전체_건수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
):
    # 1) 정렬용 컬럼 준비
    df_sorted = df.copy()
    if "기준_날짜" in time_col:
        df_sorted["기준_날짜"] = pd.to_datetime(df_sorted["기준_날짜"])

    # 2) 기준_날짜 + 시간대로 정렬
    df_sorted = df_sorted.sort_values(list(time_col)).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 3) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 4) 파이프라인 (정규화 + 랜덤포레스트)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1))
    ])

    # 5) 하이퍼파라미터 탐색
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 12],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", "log2"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # 6) 점수 계산
    def _eval(name, X_split, y_split):
        pred = best_model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": np.sqrt(mean_squared_error(y_split, pred))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test),
        "best_params": search.best_params_
    }

    return best_model, scores


In [12]:
train_model(st_2247)

(Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  RandomForestRegressor(max_depth=8, max_features='sqrt',
                                        min_samples_leaf=2, min_samples_split=5,
                                        n_estimators=400, n_jobs=-1,
                                        random_state=42))]),
 {'train': {'split': 'train',
   'r2': 0.0786477279588309,
   'mae': 0.049079611974128586,
   'rmse': np.float64(0.16249227403904126)},
  'val': {'split': 'val',
   'r2': 0.0006072259739349217,
   'mae': 0.060604605261541517,
   'rmse': np.float64(0.1928816658086755)},
  'test': {'split': 'test',
   'r2': 0.001950220060574992,
   'mae': 0.04893589828106584,
   'rmse': np.float64(0.17293431704391063)},
  'best_params': {'model__max_depth': 8,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 5,
   'model__n_estimators': 400}})

In [30]:
# HGB 버전 train_model (시간순 분할)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_model_hgb(
    df,
    time_col=("기준_날짜", "시간대"),
    feature_cols= (
    '시간대',
    'day_type',
    '온도',
    '습도',
    '불쾌지수',
    '강수량',
    '적설량',
    'lag1',
    'lag3',
    'lag24',
    'hour_sin',
    'hour_cos',
    'rolling3',
    'rolling24'
    ),
    target_col="전체_건수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
): 
    # 1) 정렬용 컬럼 준비
    df_sorted = df.copy()
    if "기준_날짜" in time_col:
        df_sorted["기준_날짜"] = pd.to_datetime(df_sorted["기준_날짜"], errors="coerce")

    # 2) 기준_날짜 + 시간대로 정렬
    df_sorted = df_sorted.sort_values(list(time_col)).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 3) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 4) HGB 모델
    model = HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=600,
        max_depth=10,
        min_samples_leaf=10,
        random_state=random_state
    )
    model.fit(X_train, y_train)

    # 5) 점수 계산
    def _eval(name, X_split, y_split):
        pred = model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": float(np.sqrt(mean_squared_error(y_split, pred)))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test)
    }

    return model, scores


In [31]:
train_model_hgb(st_2247)

(HistGradientBoostingRegressor(learning_rate=0.03, max_depth=10, max_iter=600,
                               min_samples_leaf=10, random_state=42),
 {'train': {'split': 'train',
   'r2': 0.15544838712897702,
   'mae': 0.04556657648678615,
   'rmse': 0.15557254587924496},
  'val': {'split': 'val',
   'r2': -0.008418598152222456,
   'mae': 0.06160996267397777,
   'rmse': 0.19375069498356545},
  'test': {'split': 'test',
   'r2': 0.01377765594143221,
   'mae': 0.050527963195784495,
   'rmse': 0.17190658002710577}})